# 실습 문제 모범답안 - 101. Tools & Agents

`101_Tools_Agents` 노트북 실습 문제에 대한 모범답안 예시입니다.

### 구현 요약

1. `@tool` 데코레이터로 `calc_discount` 도구 정의 (타입 힌트 + docstring)
2. Pydantic `args_schema`로 `calc_bmi` 도구 정의
3. 두 도구로 에이전트 생성 후 복합 질문으로 도구 호출 확인

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4-mini", model_provider="openai")

## 1. @tool 데코레이터로 calc_discount 도구 정의

타입 힌트가 입력 스키마가 되고, docstring이 모델에게 도구의 용도를 알려줍니다.

In [3]:
from langchain.tools import tool

@tool
def calc_discount(price: float, discount_percent: float) -> str:
    """상품의 할인가를 계산합니다.

    Args:
        price: 상품 원가 (원)
        discount_percent: 할인율 (%)
    """
    discounted = price * (1 - discount_percent / 100)
    return f"원가 {price:,.0f}원의 {discount_percent}% 할인가는 {discounted:,.0f}원입니다."

# 도구 단독 테스트
print(calc_discount.invoke({"price": 50000, "discount_percent": 20}))

원가 50,000원의 20.0% 할인가는 40,000원입니다.


C:\Users\trimu\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. Pydantic 스키마로 calc_bmi 도구 정의

`Field(description=...)`로 각 인자의 의미를 모델에게 명확히 전달합니다.

In [4]:
from pydantic import BaseModel, Field

class BmiInput(BaseModel):
    """BMI 계산에 사용할 입력 스키마"""
    height_cm: float = Field(description="키 (센티미터 단위)")
    weight_kg: float = Field(description="몸무게 (킬로그램 단위)")

@tool(args_schema=BmiInput)
def calc_bmi(height_cm, weight_kg) -> str:
    """키와 몸무게로 BMI(체질량지수)를 계산합니다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    return f"키 {height_cm}cm, 몸무게 {weight_kg}kg의 BMI는 {bmi:.1f}입니다."

# 도구 단독 테스트
print(calc_bmi.invoke({"height_cm": 175, "weight_kg": 70}))

키 175.0cm, 몸무게 70.0kg의 BMI는 22.9입니다.


## 3. 에이전트 생성 + 복합 질문으로 호출

두 도구가 모두 필요한 질문을 던지면, 에이전트가 각 도구를 알맞은 인자로 호출합니다.

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[calc_discount, calc_bmi],
)

result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "당신은 도움이 되는 어시스턴트입니다. 주어진 도구를 이용해 답변하세요."},
        {"role": "user", "content": "키 175cm 몸무게 70kg의 BMI와, 50,000원 상품의 20% 할인가를 알려줘."},
    ]}
)

# 도구 호출 과정 포함 전체 메시지 확인
for msg in result["messages"]:
    msg.pretty_print()

================================ System Message ================================

당신은 도움이 되는 어시스턴트입니다. 주어진 도구를 이용해 답변하세요.
================================ Human Message =================================

키 175cm 몸무게 70kg의 BMI와, 50,000원 상품의 20% 할인가를 알려줘.
================================== Ai Message ==================================
Tool Calls:
  calc_bmi (call_vq0FQVu4XLJtcldlHT6xQrGV)
 Call ID: call_vq0FQVu4XLJtcldlHT6xQrGV
  Args:
    height_cm: 175
    weight_kg: 70
  calc_discount (call_aR4EzfnoTuw44y8Q2FWxyKvE)
 Call ID: call_aR4EzfnoTuw44y8Q2FWxyKvE
  Args:
    price: 50000
    discount_percent: 20
================================= Tool Message =================================
Name: calc_bmi

키 175.0cm, 몸무게 70.0kg의 BMI는 22.9입니다.
================================= Tool Message =================================
Name: calc_discount

원가 50,000원의 20.0% 할인가는 40,000원입니다.
================================== Ai Message ==================================

- BMI: **22.9**
- 50,000원 상품의 20% 

## 확인 포인트

- 출력에서 에이전트가 `calc_bmi`와 `calc_discount`를 **각각 알맞은 인자**로 호출하는지 확인하세요.
- 도구의 docstring과 `Field(description=...)`이 있어야 모델이 어떤 도구를 언제 쓸지 판단할 수 있습니다.